# RAG System Analysis and Testing

This notebook demonstrates testing and analysis of the RAG (Retrieval-Augmented Generation) system components. We'll explore each module from the `src` directory, perform basic tests, and analyze the pipeline's behavior with sample data.

In [ ]:
# Import necessary libraries and modules
import sys
import os
sys.path.append('../')

from src.config_loader import load_config
from src.embedding import get_embedding_model
from src.index import create_or_update_index, load_local_index
from src.llm import get_llm
from src.rag_pipeline import get_rag_chain, generate_answer
from src.ingest import ingest_data
from src.chunking import chunk_documents
from src.retriever import get_retriever

print("Modules imported successfully!")

## Configuration Loading

First, let's load the system configuration from the YAML file. This centralizes all settings for easy modification.

In [ ]:
# Load configuration
config = load_config()
print("Configuration loaded:")
print(f"- Embedding model: {config.embedding.model_name}")
print(f"- LLM model: {config.llm.model_name}")
print(f"- Raw data directory: {config.paths.raw_data_dir}")
print(f"- Vector store directory: {config.paths.vector_store_dir}")

## Embedding Model Test

Test the embedding model initialization and basic functionality.

In [ ]:
# Initialize embedding model
embed_model = get_embedding_model(config.embedding.model_name)
print(f"Embedding model initialized: {type(embed_model).__name__}")

# Test embedding a sample text
sample_text = "This is a test document for embedding."
embedding = embed_model.embed_query(sample_text)
print(f"Sample embedding dimension: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")

## Data Ingestion and Chunking

Ingest raw documents and chunk them for processing.

In [ ]:
# Ingest data
raw_docs = ingest_data(config.paths.raw_data_dir, config.ingestion.allowed_extensions)
print(f"Ingested {len(raw_docs)} documents")

# Chunk documents
chunks = chunk_documents(
    raw_docs, 
    chunk_size=config.chunking.size, 
    chunk_overlap=config.chunking.overlap
)
print(f"Created {len(chunks)} chunks")
print(f"Sample chunk: {chunks[0].page_content[:100]}...")

## Vector Index Creation

Create or load the vector index for the knowledge base.

In [ ]:
# Check if index exists
vdb_path = config.paths.vector_store_dir
index_file = os.path.join(vdb_path, "index.faiss")

if not os.path.exists(index_file):
    print("Creating new vector index...")
    vector_db = create_or_update_index(chunks, embed_model, vdb_path)
    print("Index created successfully")
else:
    print("Loading existing vector index...")
    vector_db = load_local_index(vdb_path, embed_model)
    print("Index loaded successfully")

# Test similarity search
query_embedding = embed_model.embed_query("test query")
similar_docs = vector_db.similarity_search_by_vector(query_embedding, k=2)
print(f"Found {len(similar_docs)} similar documents for test query")

## Retriever Setup

Set up the hybrid retriever for document retrieval.

In [ ]:
# Initialize LLM for retriever (if needed)
llm = get_llm(
    provider="openai",
    model_name=config.llm.model_name,
    temperature=config.llm.temperature
)

# Set up retriever
retriever = get_retriever(vector_db, llm, chunks, k=3)
print(f"Retriever initialized: {type(retriever).__name__}")

# Test retrieval
test_query = "What is machine learning?"
retrieved_docs = retriever.get_relevant_documents(test_query)
print(f"Retrieved {len(retrieved_docs)} documents for query: '{test_query}'")
for i, doc in enumerate(retrieved_docs):
    print(f"Doc {i+1}: {doc.page_content[:50]}...")

## RAG Pipeline Testing

Assemble and test the complete RAG pipeline with sample queries.

In [ ]:
# Assemble RAG chain
rag_chain = get_rag_chain(llm, retriever)
print("RAG chain assembled successfully")

# Test with sample queries
sample_queries = [
    "Explain the concept of retrieval-augmented generation.",
    "How does vector search work?",
    "What are the benefits of using embeddings?"
]

for query in sample_queries:
    print(f"\nQuery: {query}")
    response = generate_answer(rag_chain, query)
    print(f"Response: {response}")
    print("-" * 50)

## Analysis of Results

Perform some basic analysis on the system's performance and outputs.

In [ ]:
# Analyze chunk statistics
chunk_lengths = [len(chunk.page_content) for chunk in chunks]
avg_chunk_length = sum(chunk_lengths) / len(chunk_lengths)
print(f"Average chunk length: {avg_chunk_length:.0f} characters")
print(f"Min chunk length: {min(chunk_lengths)}")
print(f"Max chunk length: {max(chunk_length)}")

# Test response times (basic)
import time
start_time = time.time()
response = generate_answer(rag_chain, "Quick test query")
end_time = time.time()
print(f"Response time for simple query: {end_time - start_time:.2f} seconds")

print("\nNotebook analysis complete!")